# Phase 5b — RoBERTa: Two-Strategy Comparison

Δύο στρατηγικές σε ένα notebook:

| Στρατηγική | Δεδομένα | Epochs | Submission |
|---|---|---|---|
| A | train-only + early stopping (3 epochs max) | ≤3 | `submission_roberta_A.csv` |
| B | train+valid, 3 σταθερά epochs | 3 | `submission_roberta_B.csv` |

Υποβάλλουμε και τα δύο στο Kaggle και κρατάμε το καλύτερο.

In [ ]:
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizer, RobertaForSequenceClassification
from torch.optim import AdamW
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score
import copy
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
train = pd.read_csv('/train.csv')
valid = pd.read_csv('/valid.csv')
test  = pd.read_csv('/test.csv')
train_full = pd.concat([train, valid], ignore_index=True)

def make_text(df):
    return (df['title'].fillna('') + ' ' + df['text'].fillna('').str[:550]).tolist()

texts_train      = make_text(train)
texts_train_full = make_text(train_full)
texts_valid      = make_text(valid)
texts_test       = make_text(test)

print(f'Train only:  {len(texts_train)}')
print(f'Train+Valid: {len(texts_train_full)}')
print(f'Valid:       {len(texts_valid)}')
print(f'Test:        {len(texts_test)}')

In [ ]:
# Label Encoding — fit σε train_full για να δει όλες τις κλάσεις
le_hazard  = LabelEncoder()
le_product = LabelEncoder()
le_hazard.fit(train_full['hazard-category'])
le_product.fit(train_full['product-category'])

# Labels για κάθε split
y_train_hazard       = le_hazard.transform(train['hazard-category'])
y_train_product      = le_product.transform(train['product-category'])
y_train_full_hazard  = le_hazard.transform(train_full['hazard-category'])
y_train_full_product = le_product.transform(train_full['product-category'])
y_valid_hazard       = le_hazard.transform(valid['hazard-category'])
y_valid_product      = le_product.transform(valid['product-category'])

n_hazard  = len(le_hazard.classes_)
n_product = len(le_product.classes_)
print(f'Hazard classes: {n_hazard}, Product classes: {n_product}')

In [ ]:
class FoodHazardDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts      = texts
        self.labels     = labels
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      encoded['input_ids'].squeeze(),
            'attention_mask': encoded['attention_mask'].squeeze(),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }


def train_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0
    for batch in loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)
        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        outputs.loss.backward()
        optimizer.step()
        total_loss += outputs.loss.item()
    return total_loss / len(loader)


def get_predictions(model, loader):
    model.eval()
    all_preds = []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            preds = model(input_ids=input_ids, attention_mask=attention_mask).logits.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
    return np.array(all_preds)


def get_probabilities(model, loader):
    model.eval()
    all_probs = []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            probs = torch.softmax(model(input_ids=input_ids, attention_mask=attention_mask).logits, dim=1).cpu().numpy()
            all_probs.append(probs)
    return np.vstack(all_probs)


def official_st1_score(y_true_hazard, y_pred_hazard,
                       y_true_product, y_pred_product, verbose=True):
    f1_hazard = f1_score(y_true_hazard, y_pred_hazard, average='macro', zero_division=0)
    mask = (np.array(y_true_hazard) == np.array(y_pred_hazard))
    f1_product = f1_score(np.array(y_true_product)[mask],
                          np.array(y_pred_product)[mask],
                          average='macro', zero_division=0) if mask.sum() > 0 else 0.0
    score = (f1_hazard + f1_product) / 2
    if verbose:
        print(f'macro-F1 Hazard:                    {f1_hazard:.4f}')
        print(f'Σωστά hazard predictions:           {mask.sum()}/{len(mask)} ({100*mask.mean():.1f}%)')
        print(f'macro-F1 Product (given correct h): {f1_product:.4f}')
        print(f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
        print(f'OFFICIAL ST1 SCORE:                 {score:.4f}')
    return score

In [ ]:
MODEL_NAME = 'roberta-base'
BATCH_SIZE = 32
MAX_LENGTH = 128
LR         = 2e-5
MAX_EPOCHS = 3   # 3 epochs — αρκετό για RoBERTa

tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)
print('Tokenizer έτοιμος!')

dummy_labels = np.zeros(len(texts_test), dtype=int)

## Στρατηγική Α — Train-only + Early Stopping (≤3 epochs)

Εκπαίδευση μόνο στο train set.
Χρησιμοποιούμε το validation για early stopping,
αλλά με max 3 epochs για να περιορίσουμε το overfitting.

In [ ]:
print('=== ΣΤΡΑΤΗΓΙΚΗ Α — HAZARD (train-only + early stopping) ===\n')

# DataLoaders για Στρατηγική Α
train_loader_A_hazard = DataLoader(
    FoodHazardDataset(texts_train, y_train_hazard, tokenizer, MAX_LENGTH),
    batch_size=BATCH_SIZE, shuffle=True)
valid_loader_hazard = DataLoader(
    FoodHazardDataset(texts_valid, y_valid_hazard, tokenizer, MAX_LENGTH),
    batch_size=BATCH_SIZE, shuffle=False)
test_loader_hazard = DataLoader(
    FoodHazardDataset(texts_test, dummy_labels, tokenizer, MAX_LENGTH),
    batch_size=BATCH_SIZE, shuffle=False)

model_A_hazard = RobertaForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=n_hazard).to(device)
optimizer_A_hazard = AdamW(model_A_hazard.parameters(), lr=LR)

best_f1_A_hazard   = 0
best_state_A_hazard = None

for epoch in range(MAX_EPOCHS):
    loss = train_epoch(model_A_hazard, train_loader_A_hazard, optimizer_A_hazard)
    preds = le_hazard.inverse_transform(get_predictions(model_A_hazard, valid_loader_hazard))
    f1 = f1_score(valid['hazard-category'], preds, average='macro', zero_division=0)
    print(f'Epoch {epoch+1}/{MAX_EPOCHS} | Loss: {loss:.4f} | Valid F1 Hazard: {f1:.4f}', end='')
    if f1 > best_f1_A_hazard:
        best_f1_A_hazard    = f1
        best_state_A_hazard = copy.deepcopy(model_A_hazard.state_dict())
        print(' ✓ (αποθηκεύτηκε)')
    else:
        print(' (δεν βελτιώθηκε)')

model_A_hazard.load_state_dict(best_state_A_hazard)
print(f'\nΚαλύτερο epoch: F1={best_f1_A_hazard:.4f}')

In [ ]:
print('=== ΣΤΡΑΤΗΓΙΚΗ Α — PRODUCT (train-only + early stopping) ===\n')

train_loader_A_product = DataLoader(
    FoodHazardDataset(texts_train, y_train_product, tokenizer, MAX_LENGTH),
    batch_size=BATCH_SIZE, shuffle=True)
valid_loader_product = DataLoader(
    FoodHazardDataset(texts_valid, y_valid_product, tokenizer, MAX_LENGTH),
    batch_size=BATCH_SIZE, shuffle=False)
test_loader_product = DataLoader(
    FoodHazardDataset(texts_test, dummy_labels, tokenizer, MAX_LENGTH),
    batch_size=BATCH_SIZE, shuffle=False)

model_A_product = RobertaForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=n_product).to(device)
optimizer_A_product = AdamW(model_A_product.parameters(), lr=LR)

best_f1_A_product    = 0
best_state_A_product = None

for epoch in range(MAX_EPOCHS):
    loss = train_epoch(model_A_product, train_loader_A_product, optimizer_A_product)
    preds = le_product.inverse_transform(get_predictions(model_A_product, valid_loader_product))
    f1 = f1_score(valid['product-category'], preds, average='macro', zero_division=0)
    print(f'Epoch {epoch+1}/{MAX_EPOCHS} | Loss: {loss:.4f} | Valid F1 Product: {f1:.4f}', end='')
    if f1 > best_f1_A_product:
        best_f1_A_product    = f1
        best_state_A_product = copy.deepcopy(model_A_product.state_dict())
        print(' ✓ (αποθηκεύτηκε)')
    else:
        print(' (δεν βελτιώθηκε)')

model_A_product.load_state_dict(best_state_A_product)
print(f'\nΚαλύτερο epoch: F1={best_f1_A_product:.4f}')

In [ ]:
# Αξιολόγηση Στρατηγικής Α
preds_A_hazard  = le_hazard.inverse_transform(get_predictions(model_A_hazard,  valid_loader_hazard))
preds_A_product = le_product.inverse_transform(get_predictions(model_A_product, valid_loader_product))

print('=== ΑΞΙΟΛΟΓΗΣΗ ΣΤΡΑΤΗΓΙΚΗΣ Α (validation) ===')
print('(Προσοχή: early stopping βασίστηκε σε αυτό το set — score overestimated)\n')
score_A = official_st1_score(
    valid['hazard-category'], preds_A_hazard,
    valid['product-category'], preds_A_product
)

# Submission Α
test_hazard_A  = le_hazard.inverse_transform(get_predictions(model_A_hazard,  test_loader_hazard))
test_product_A = le_product.inverse_transform(get_predictions(model_A_product, test_loader_product))

pd.DataFrame({'id': test['id'], 'hazard-category': test_hazard_A, 'product-category': test_product_A}) \
  .to_csv('submission_roberta_A.csv', index=False)
print('\nΑποθηκεύτηκε: submission_roberta_A.csv')

# Αποθήκευση probabilities για ensemble
np.save('roberta_A_valid_hazard_probs.npy',  get_probabilities(model_A_hazard,  valid_loader_hazard))
np.save('roberta_A_valid_product_probs.npy', get_probabilities(model_A_product, valid_loader_product))
np.save('roberta_A_test_hazard_probs.npy',   get_probabilities(model_A_hazard,  test_loader_hazard))
np.save('roberta_A_test_product_probs.npy',  get_probabilities(model_A_product, test_loader_product))
print('Probabilities αποθηκεύτηκαν (για ensemble)')

## Στρατηγική Β — Train+Valid, 3 Σταθερά Epochs

Εκπαίδευση σε όλα τα δεδομένα, χωρίς early stopping.
Το validation score εδώ είναι πάντα overestimated
αφού το μοντέλο έχει δει αυτά τα δεδομένα.
Το αληθινό score φαίνεται μόνο στο Kaggle.

In [ ]:
print('=== ΣΤΡΑΤΗΓΙΚΗ Β — HAZARD (train+valid, 3 epochs) ===\n')

train_loader_B_hazard = DataLoader(
    FoodHazardDataset(texts_train_full, y_train_full_hazard, tokenizer, MAX_LENGTH),
    batch_size=BATCH_SIZE, shuffle=True)

model_B_hazard = RobertaForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=n_hazard).to(device)
optimizer_B_hazard = AdamW(model_B_hazard.parameters(), lr=LR)

for epoch in range(MAX_EPOCHS):
    loss = train_epoch(model_B_hazard, train_loader_B_hazard, optimizer_B_hazard)
    print(f'Epoch {epoch+1}/{MAX_EPOCHS} | Loss: {loss:.4f}')

torch.save(model_B_hazard.state_dict(), 'roberta_B_hazard.pt')
print('Αποθηκεύτηκε: roberta_B_hazard.pt')

In [ ]:
print('=== ΣΤΡΑΤΗΓΙΚΗ Β — PRODUCT (train+valid, 3 epochs) ===\n')

train_loader_B_product = DataLoader(
    FoodHazardDataset(texts_train_full, y_train_full_product, tokenizer, MAX_LENGTH),
    batch_size=BATCH_SIZE, shuffle=True)

model_B_product = RobertaForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=n_product).to(device)
optimizer_B_product = AdamW(model_B_product.parameters(), lr=LR)

for epoch in range(MAX_EPOCHS):
    loss = train_epoch(model_B_product, train_loader_B_product, optimizer_B_product)
    print(f'Epoch {epoch+1}/{MAX_EPOCHS} | Loss: {loss:.4f}')

torch.save(model_B_product.state_dict(), 'roberta_B_product.pt')
print('Αποθηκεύτηκε: roberta_B_product.pt')

In [ ]:
# Submission Β
test_hazard_B  = le_hazard.inverse_transform(get_predictions(model_B_hazard,  test_loader_hazard))
test_product_B = le_product.inverse_transform(get_predictions(model_B_product, test_loader_product))

pd.DataFrame({'id': test['id'], 'hazard-category': test_hazard_B, 'product-category': test_product_B}) \
  .to_csv('submission_roberta_B.csv', index=False)
print('Αποθηκεύτηκε: submission_roberta_B.csv')

# Probabilities για ensemble
np.save('roberta_B_test_hazard_probs.npy',   get_probabilities(model_B_hazard,  test_loader_hazard))
np.save('roberta_B_test_product_probs.npy',  get_probabilities(model_B_product, test_loader_product))


## Σύνοψη

| Submission | Στρατηγική |
|---|---|
| `submission_roberta_A.csv` | train-only + early stopping (≤3 epochs) |
| `submission_roberta_B.csv` | train+valid, 3 σταθερά epochs |


Αρχεία για ensemble:
- `roberta_A_valid_hazard_probs.npy` / `roberta_A_valid_product_probs.npy`
- `roberta_A_test_hazard_probs.npy` / `roberta_A_test_product_probs.npy`
- `roberta_B_test_hazard_probs.npy` / `roberta_B_test_product_probs.npy`